In [1]:

import torch, torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
import time
import os
import cv2
from dataPreparation import dataPrep
from imutils import paths
from sklearn.model_selection import train_test_split
from torch.nn import Module

In [2]:

dataset_path = "/Users/beyzaecemerce/Desktop/GitHub/thesis/drone"
image_path=dataset_path+'/dataset/semantic_drone_dataset/original_images'
masked_path = dataset_path+ '/RGB_color_image_masks/RGB_color_image_masks'
TEST_SPLIT = 0.20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = True if DEVICE == "cuda" else False

In [3]:
image = cv2.imread(image_path+"/100.jpg")
mask=cv2.imread(masked_path+"/100.png")
print(image.shape)
print(mask.shape)
figure, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))
ax[0].grid(False)
ax[1].grid(False)
ax[0].imshow(image)
ax[1].imshow(mask)
ax[0].set_title("Image")
ax[1].set_title("Original Mask")
figure.tight_layout()
figure.show()

(4000, 6000, 3)
(4000, 6000, 3)


/var/folders/xy/y85s_w552q1_8cd2gzzrqfmc0000gn/T/ipykernel_97421/754042599.py:13: UserWarning: Matplotlib is currently using module://matplotlib_inline.backend_inline, which is a non-GUI backend, so cannot show the figure.
  figure.show()


In [3]:
INIT_LR = 2e-4
NUM_EPOCHS = 300
BATCH_SIZE = 8

INPUT_IMAGE_WIDTH = 6000
INPUT_IMAGE_HEIGHT = 4000

BASE_OUTPUT = dataset_path+"/output"

MODEL_PATH = os.path.join(BASE_OUTPUT, "MODEL.pth")
PLOT_PATH = os.path.sep.join([BASE_OUTPUT, "Train_Test_Plot.png"])
TEST_PATHS = os.path.sep.join([BASE_OUTPUT, "test_path.txt"])

In [4]:
imagePaths = sorted(list(paths.list_images(image_path)))
maskPaths = sorted(list(paths.list_images(masked_path)))

split = train_test_split(imagePaths, maskPaths,test_size=TEST_SPLIT, random_state=42)

(trainImages, testImages) = split[:2]
(trainMasks, testMasks) = split[2:]

print("[INFO] saving testing image paths...")
f = open(TEST_PATHS, "w")
f.write("\n".join(testImages))
f.close()

[INFO] saving testing image paths...


In [5]:
input_transform = transforms.Compose([
        transforms.Resize((200, 300),interpolation=cv2.INTER_NEAREST),  # Resize the input image to (height=256, width=384)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
mask_transform = transforms.Compose([
    transforms.Resize((256, 384)),  # Resize the mask to the same size as the input image
    transforms.ToTensor()
])

In [7]:
train = dataPrep(imagePaths=trainImages, maskPaths=trainMasks, input_transform=input_transform, mask_transform=mask_transform)
test = dataPrep(imagePaths=testImages, maskPaths=testMasks, input_transform=input_transform, mask_transform=mask_transform)

In [8]:
from torch.nn import CrossEntropyLoss
from torch.optim import Adam